# Intro

In [123]:
# filter warnings
import warnings
warnings.filterwarnings('ignore')

# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# change plotly style
import plotly.io as pio

#a list of available styles
#pio.templates
# change the style
pio.templates.default = "presentation"

In [124]:
# generate some linear-looking data
np.random.seed(42)  # to make this code example reproducible
m = 100  # number of instances
X = 2 * np.random.rand(m, 1)  # column vector
y = 4 + 3 * X + np.random.randn(m, 1)  # column vector

df = pd.DataFrame(np.hstack((X, y)), columns=['X', 'y'])
df.head()

,X,y
0,0.749080,6.334288
1,1.901429,9.405278
2,1.463988,8.483724
3,1.197317,5.604382
4,0.312037,4.716440


In [125]:
# plot the data
fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Linear-looking data', labels={'x': 'X', 'y': 'y'}, width=600, height=400)
fig.show()

In [126]:
df.corr()

,X,y
X,1.000000,0.877082
y,0.877082,1.000000


# Linear Regression (OLS)

In [127]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X, y)

LinearRegression()

In [128]:
lin_reg.intercept_

array([4.21509616])

In [129]:
lin_reg.coef_

array([[2.77011339]])

In [130]:
# Let's look at the model parameters θ that minimize the cost function.

intercept = round(lin_reg.intercept_[0], 1)
slope = round(lin_reg.coef_[0][0], 1)
print(f'intercept: {intercept}, slope: {slope}')
print(f'equation: y = {intercept} + {slope} * X')

intercept: 4.2, slope: 2.8
equation: y = 4.2 + 2.8 * X


In [131]:
y_pred = lin_reg.predict(X)

fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Linear-looking data', labels={'x': 'X', 'y': 'y'}, width=800, height=600)

# add the predicted values to the plot (as a line with diffferent color)
fig.add_scatter(x=X.flatten(), y=y_pred.flatten(), mode='lines', name='Predictions')
fig.show()

In [132]:
# create different lines with different values of the intercept and slope
intercept_lst = [ 3.5, 4.1, 4.5]
slope_lst = [ 3, 2.7, 2.9]

fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Linear-looking data', labels={'x': 'X', 'y': 'y'}, width=1000, height=600)

# add the predicted values to the plot (as a line with diffferent color)
fig.add_scatter(x=X.flatten(), y=y_pred.flatten(), mode='lines', name='y = {} + {} * X'.format(intercept , slope))

# add the different lines to the plot
for i in range(len(intercept_lst)):
    y_new = intercept_lst[i] + slope_lst[i] * X
    fig.add_scatter(x=X.flatten(), y=y_new.flatten(), mode='lines', name='y = {} + {} * X'.format(intercept_lst[i], slope_lst[i]))
fig.show()

In [133]:
# create different lines with different values of the intercept and slope
y1 = 4.2 + 2.8 * X
y2 = 3.5 + 3 * X
y3 = 4.1 + 2.7 * X
y4 = 4.5 + 2.9 * X

In [134]:
# Calculate the RMSE for each line
from sklearn.metrics import mean_squared_error

rmse1 = np.sqrt(mean_squared_error(y, y1))
rmse2 = np.sqrt(mean_squared_error(y, y2))
rmse3 = np.sqrt(mean_squared_error(y, y3))
rmse4 = np.sqrt(mean_squared_error(y, y4))

print('RMSE1: ', rmse1)
print('RMSE2: ', rmse2)
print('RMSE3: ', rmse3)
print('RMSE4: ', rmse4)

RMSE1:  0.8983689708289428
RMSE2:  1.0363524592244577
RMSE3:  0.9171033489200907
RMSE4:  0.9890308374828696


# Normal Equation from Scratch

In [135]:
# use the normal equation to find the best parameters

X_b = np.c_[np.ones((m, 1)), X]  # add x0 = 1 to each instance
weight_best = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
weight_best 

array([[4.21509616],
       [2.77011339]])

In [136]:
# X_b

In [137]:
intercept = round(weight_best[0][0], 1)
slope = round(weight_best[1][0], 1)

fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Linear-looking data', labels={'x': 'X', 'y': 'y'}, width=1000, height=600)

# add the predicted values to the plot (as a line with diffferent color)
fig.add_scatter(x=X.flatten(), y=y_pred.flatten(), mode='lines', name='y = {} + {} * X'.format(intercept , slope))

# Gradient Descent

In [138]:
X = df ['X'].values.reshape(-1, 1)
X_b = np.c_[np.ones((m, 1)), X]  # add x0 = 1 to each instance
y = df ['y'].values.reshape(-1, 1)

In [139]:
X_b.shape, y.shape

((100, 2), (100, 1))

In [140]:
# Batch Gradient Descent

eta = 0.01  # learning rate
n_iterations = 1000
m = len(X)  # number of instances

np.random.seed(42)
weights = np.random.randn(2, 1)  # randomly initialized model parameters

for iteration in range(n_iterations):
    gradients = 2 / m * X_b.T @ (X_b @ weights - y)   # dj/dw0, dj/dw1
    weights = weights - eta * gradients

In [141]:
weights

array([[4.1935218 ],
       [2.78916237]])

In [142]:
# Stochastic Gradient Descent

from sklearn.linear_model import SGDRegressor

lr = SGDRegressor(max_iter=10000, tol=1e-9, penalty=None, eta0=0.01, random_state=0, learning_rate='constant')
lr.fit(X, y)

lr.intercept_, lr.coef_

(array([4.23515905]), array([2.81696233]))

In [143]:
y_pred = lr.predict(X)
fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Linear-looking data', labels={'x': 'X', 'y': 'y'}, width=1000, height=600)

# add the predicted values to the plot (as a line with diffferent color)
fig.add_scatter(x=X.flatten(), y=y_pred.flatten(), mode='lines', name='y = {} + {} * X'.format(intercept , slope))

# Polynomial Regression


In [144]:
np.random.seed(42)
m = 100
X = 6 * np.random.rand(m, 1) - 3
y = 0.5 * X ** 2 + X + 2 + np.random.randn(m, 1)

In [145]:
px.scatter(x=X.flatten(), y=y.flatten(), title='Quadratic-looking data', labels={'x': 'X', 'y': 'y'}, width=800, height=500)

In [146]:
# Apply Linear Regression

lin_reg = LinearRegression()
lin_reg.fit(X, y)
lin_reg.intercept_, lin_reg.coef_

(array([3.56401543]), array([[0.84362064]]))

In [147]:
# Plot the predictions

X_new = np.linspace(-3, 3, 100).reshape(100, 1)
y_new = lin_reg.predict(X_new)

fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Quadratic-looking data', labels={'x': 'X', 'y': 'y'}, width=800, height=500)
fig.add_scatter(x=X_new.flatten(), y=y_new.flatten(), mode='lines', name='Predictions')
fig.show()

In [148]:
from sklearn.preprocessing import PolynomialFeatures

poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly_features.fit_transform(X)

In [149]:
lin_reg = LinearRegression()
lin_reg.fit(X_poly, y)
lin_reg.intercept_, lin_reg.coef_

(array([1.78134581]), array([[0.93366893, 0.56456263]]))

In [150]:
# plot the data
fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Quadratic-looking data', labels={'x': 'X', 'y': 'y'}, width=800, height=500)

X_new = np.linspace(-3, 3, 100).reshape(100, 1)
X_new_poly = poly_features.transform(X_new)
y_new = lin_reg.predict(X_new_poly)

# add the predicted values to the plot (as a line with diffferent color)
fig.add_scatter(x=X_new.flatten(), y=y_new.flatten(), mode='lines', name='Predictions')
fig.show()

# Regularized Linear Regression

In [151]:
# Ridge Regression

from sklearn.linear_model import Ridge

ridge_reg = Ridge(alpha=1)
ridge_reg.fit(X, y)
print("Ridge Regression: ", ridge_reg.predict([[1.5]]))

sgd_reg = SGDRegressor(max_iter=1000, tol=1e-5, penalty="l2", random_state=42)
sgd_reg.fit(X, y.ravel())
print("SGD Regression: ", sgd_reg.predict([[1.5]]))


Ridge Regression:  [[4.82497007]]
SGD Regression:  [4.81978491]


In [152]:
# Lasso Regression

from sklearn.linear_model import Lasso

lasso_reg = Lasso(alpha=0.1)
lasso_reg.fit(X, y)
print("Lasso Regression: ", lasso_reg.predict([[1.5]]))

sgd_reg = SGDRegressor(max_iter=1000, tol=1e-5, penalty="l1", random_state=42)
sgd_reg.fit(X, y.ravel())
print("SGD Regression: ", sgd_reg.predict([[1.5]]))

Lasso Regression:  [4.77621741]
SGD Regression:  [4.8197765]


In [153]:
#  plot Ridge and Lasso Regression


fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Quadratic-looking data', labels={'x': 'X', 'y': 'y'}, width=800, height=500)

X_new = np.linspace(-3, 3, 100).reshape(100, 1)

y_new = ridge_reg.predict(X_new)
fig.add_scatter(x=X_new.flatten(), y=y_new.flatten(), mode='lines', name='Ridge Regression', line=dict(color='yellow'))

y_new = lasso_reg.predict(X_new)
fig.add_scatter(x=X_new.flatten(), y=y_new.flatten(), mode='lines', name='Lasso Regression', line=dict(color='black'))

fig.show()

In [154]:
# Polynomial Regression with Ridge and Lasso Regression

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

poly_reg = Pipeline([ 
    ("poly_features", PolynomialFeatures(degree=10, include_bias=False)),
    ("std_scaler", StandardScaler()),
    ("lin_reg", LinearRegression()),
])

poly_reg.fit(X, y)
print("Polynomial Regression: ", poly_reg.predict([[1.5]]))


Polynomial Regression:  [[4.44851584]]


In [155]:
# plot the data
fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Quadratic-looking data', labels={'x': 'X', 'y': 'y'}, width=800, height=500)

X_new = np.linspace(-3, 3, 100).reshape(100, 1)
y_new = poly_reg.predict(X_new)

# add the predicted values to the plot (as a line with diffferent color)
fig.add_scatter(x=X_new.flatten(), y=y_new.flatten(), mode='lines', name='Polynomial Regression', line=dict(color='red'))
fig.show()

In [156]:
# Polynomial Regression with Ridge

poly_reg = Pipeline([ 
    ("poly_features", PolynomialFeatures(degree=10, include_bias=False)),
    ("std_scaler", StandardScaler()),
    ("ridge_reg", Ridge(alpha=0.1)),
])

poly_reg.fit(X, y)

# plot the data
fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Quadratic-looking data', labels={'x': 'X', 'y': 'y'}, width=800, height=500)

X_new = np.linspace(-3, 3, 100).reshape(100, 1)
y_new = poly_reg.predict(X_new)

# add the predicted values to the plot (as a line with diffferent color)
fig.add_scatter(x=X_new.flatten(), y=y_new.flatten(), mode='lines', name='Polynomial Regression', line=dict(color='red'))
fig.show()

In [157]:
# Polynomial Regression with Lasso

poly_reg = Pipeline([
    ("poly_features", PolynomialFeatures(degree=10, include_bias=False)),
    ("std_scaler", StandardScaler()),
    ("lasso_reg", Lasso(alpha=0.1)),
])

poly_reg.fit(X, y)

# plot the data
fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Quadratic-looking data', labels={'x': 'X', 'y': 'y'}, width=800, height=500)

X_new = np.linspace(-3, 3, 100).reshape(100, 1)
y_new = poly_reg.predict(X_new)

# add the predicted values to the plot (as a line with diffferent color)
fig.add_scatter(x=X_new.flatten(), y=y_new.flatten(), mode='lines', name='Polynomial Regression', line=dict(color='red'))
fig.show()

In [158]:
# 